# OpenCV face detection: USB webcam to HDMI monitor

Finds faces (and eyes) in the webcam picture with OpenCV and shows the result on an HDMI
monitor, like the EDGE ZYNQ face detection demo video.

**You need**
* a USB webcam on the USB host connector **J3**, with jumpers **J6 and J7 fitted** (host mode)
* an HDMI monitor on the HDMI connector (connect it before you run this notebook)

## 1. Load the overlay and start the HDMI output

In [ ]:
from pynq.overlays.base import BaseOverlay
from pynq.lib.video import VideoMode, PIXEL_BGR
base = BaseOverlay("base.bit")

hdmi_out = base.video.hdmi_out
hdmi_out.configure(VideoMode(640, 480, 24), PIXEL_BGR)
hdmi_out.start();

## 2. Open the webcam

In [ ]:
import os
os.environ["OPENCV_LOG_LEVEL"] = "SILENT"
import cv2

cam = cv2.VideoCapture(0, cv2.CAP_V4L2)
cam.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cam.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
print("Webcam open:", cam.isOpened())
if not cam.isOpened():
    print("Check the webcam is on J3 and jumpers J6 and J7 are fitted, then run: !ls /dev/video*")

def grab():
    """One webcam picture as 640x480 BGR (some webcams ignore the size request)."""
    ok, img = cam.read()
    if ok and img.shape[:2] != (480, 640):
        img = cv2.resize(img, (640, 480))
    return ok, img

## 3. One picture: webcam to HDMI and to this notebook

In [ ]:
ok, frame = grab()
if not ok:
    raise RuntimeError("could not read from the webcam")
out = hdmi_out.newframe()
out[:] = frame
hdmi_out.writeframe(out)

%matplotlib inline
from matplotlib import pyplot as plt
plt.imshow(frame[:, :, ::-1])
plt.axis("off")
plt.show()

## 4. Face and eye detection on that picture

In [ ]:
CASCADES = "/usr/share/opencv4/haarcascades/"          # trained face and eye detectors
if not os.path.isdir(CASCADES) and hasattr(cv2, "data"):   # pip builds of OpenCV keep them here
    CASCADES = cv2.data.haarcascades
face_cascade = cv2.CascadeClassifier(CASCADES + "haarcascade_frontalface_default.xml")
eye_cascade = cv2.CascadeClassifier(CASCADES + "haarcascade_eye.xml")
assert not face_cascade.empty() and not eye_cascade.empty(), "Haar cascade files not found"

def detect(img):
    """Draw a blue box round each face and green boxes round the eyes; returns the face count."""
    small = cv2.resize(img, (320, 240))                  # detect on a smaller picture: faster
    gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.2, 5, minSize=(30, 30))
    for (x, y, w, h) in faces:
        x, y, w, h = 2 * x, 2 * y, 2 * w, 2 * h
        cv2.rectangle(img, (x, y), (x + w, y + h), (255, 0, 0), 2)
        roi = cv2.cvtColor(img[y:y + h // 2 + h // 8, x:x + w], cv2.COLOR_BGR2GRAY)
        for (ex, ey, ew, eh) in eye_cascade.detectMultiScale(roi, 1.1, 6):
            cv2.rectangle(img, (x + ex, y + ey), (x + ex + ew, y + ey + eh), (0, 255, 0), 2)
    return len(faces)

n = detect(frame)
print(n, "face(s) found")
out = hdmi_out.newframe()
out[:] = frame
hdmi_out.writeframe(out)
plt.imshow(frame[:, :, ::-1])
plt.axis("off")
plt.show()

## 5. Live face detection

Runs until you press **push button 0** (or for 2 minutes). The 7-segment display shows the
number of faces found.

In [ ]:
import time
frames, start = 0, time.time()
while time.time() - start < 120 and not base.buttons[0].read():
    ok, frame = grab()
    if not ok:
        continue
    n = detect(frame)
    base.seven_seg.show_number(n)
    out = hdmi_out.newframe()
    out[:] = frame
    hdmi_out.writeframe(out)
    frames += 1
print("{:.1f} frames per second".format(frames / (time.time() - start)))

## 6. Release the webcam and stop the HDMI output

In [ ]:
cam.release()
hdmi_out.close()
base.seven_seg.clear()